<a href="https://colab.research.google.com/github/M-Abbi/Probability-Statistics-Bootcamp/blob/main/Binomial_%26_Multinomial_Coefficient_Example_%2B_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The Scenario: The "Tech Trio" Auto-Callable Note

Let's imagine a Trading desk is structuring a 3-month exotic note linked to three major tech stocks: Apple (AAPL), Nvidia (NVDA), and Microsoft (MSFT). To price this product and manage the bank's risk, the trading desk models the stock movements at the end of each month.

---

### Part 1: The Simple Example (Binomial Coefficient)

Let's simplify first. Assume we are only looking at **AAPL** over the next 3 months. At the end of each month, AAPL can either go **Up (U)** or **Down (D)**.

If a trader wants to know the probability of AAPL ending the 3-month period with exactly **2 Up months and 1 Down month**, they don't need to manually map out every path (UUD, UDU, DUU). They use the binomial coefficient.

#### The Math
The number of distinct paths that lead to this specific outcome is given by the binomial coefficient $\binom{n}{k}$, where $n$ is the total months and $k$ is the number of Up months:

$$\binom{3}{2} = \frac{3!}{2!(3-2)!} = 3 \text{ paths}$$

If the desk estimates the probability of an Up month is $p = 0.6$ and Down is $1-p = 0.4$, the probability of this specific state hitting at month 3 is:

$$P(\text{2U, 1D}) = \binom{3}{2} \times (0.6)^2 \times (0.4)^1 = 3 \times 0.36 \times 0.4 = 43.2\%$$

The trader uses this to calculate the expected payout of a simple derivative.

---

### Part 2: The Semi-Complex Case (Multinomial Coefficient)

In the real Global Markets world, a simple "Up or Down" binary is often too crude. Stocks don't just go up or down; they can skyrocket, stagnate, or crash.

Let's look at our **Tech Trio (AAPL, NVDA, MSFT)** at the end of a crucial trading quarter (3 months from now). The trading desk categorizes the market regime into **three distinct outcomes** based on macroeconomic factors (e.g., Fed rate decisions, AI earnings):

* **Bull Case (B):** Tech rallies hard. (Probability $p_1 = 0.3$)
* **Sideways Case (S):** Market chops horizontally, low volatility. (Probability $p_2 = 0.5$)
* **Bear Case (X):** Tech sell-off. (Probability $p_3 = 0.2$)

The structured products desk is selling a note that pays a massive coupon **only if**, over the next 6 months, the market experiences exactly **3 Bull months, 2 Sideways months, and 1 Bear month**.

To price this note accurately, the quant traders need to know: *How many different ways can this specific 6-month sequence occur, and what is the total probability?*

#### The Math
Because we have more than two outcomes ($B, S, X$), the binomial coefficient expands into the **multinomial coefficient**. The formula to find the number of unique paths is:

$$\frac{n!}{k_1! \cdot k_2! \cdot k_3!}$$

Where:
* $n = 6$ (total months)
* $k_1 = 3$ (Bull months)
* $k_2 = 2$ (Sideways months)
* $k_3 = 1$ (Bear month)

Plugging in our numbers:

$$\frac{6!}{3! \cdot 2! \cdot 1!} = \frac{720}{6 \cdot 2 \cdot 1} = 60 \text{ unique paths}$$

#### Calculating the Total Probability for Pricing
Now the trader calculates the exact probability of this regime combination happening to discount the future payout to present value:

$$P = 60 \times (p_1)^3 \times (p_2)^2 \times (p_3)^1$$

$$P = 60 \times (0.3)^3 \times (0.5)^2 \times (0.2)^1$$

$$P = 60 \times 0.027 \times 0.25 \times 0.2 = 0.081 \text{ or } 8.1\%$$

---

### How the IB Desk Uses This

* **Pricing the Structure:** If the note pays out \$1,000,000 in this specific scenario, the desk knows the mathematical expectation of that payout is $8.1\% \times \$1,000,000 = \$81,000$. They will charge the client this amount, plus a structuring fee (margin).
* **Delta Hedging:** The desk doesn't just sit on this risk. If day 1 is a "Bull month," the probabilities shift. The remaining periods drop from 6 to 5, and the required combinations change. The quants recalculate the multinomial probabilities in real-time to adjust their stock hedges (buying or selling underlying shares of AAPL, NVDA, and MSFT) to remain risk-neutral.

In [1]:
import math
import random

# --- PARAMETERS FROM OUR IB GLOBAL MARKETS SCENARIO ---
# Market Regimes: [Bull (B), Sideways (S), Bear (X)]
PROBABILITIES = [0.3, 0.5, 0.2]  # Must sum to 1.0
TARGET_OUTCOMES = [3, 2, 1]      # We want exactly 3 Bull, 2 Sideways, 1 Bear months
TOTAL_MONTHS = sum(TARGET_OUTCOMES)  # 6 months

# --- 1. EXACT MATHEMATICAL PRICING (Using Multinomial Coefficient) ---

def multinomial_coefficient(n, counts):
    """Calculates n! / (k1! * k2! * ... * km!)"""
    denominator = 1
    for k in counts:
        denominator *= math.factorial(k)
    return math.factorial(n) // denominator

def calculate_exact_probability(n, counts, probs):
    """Calculates the exact probability of a specific multinomial outcome."""
    coeff = multinomial_coefficient(n, counts)

    # Calculate (p1^k1) * (p2^k2) * ... * (pm^km)
    prob_product = 1.0
    for k, p in zip(counts, probs):
        prob_product *= (p ** k)

    return coeff * prob_product

exact_prob = calculate_exact_probability(TOTAL_MONTHS, TARGET_OUTCOMES, PROBABILITIES)

print("--- EXPLICIT MATHEMATICAL PRICING ---")
print(f"Multinomial Coefficient (Unique Paths): {multinomial_coefficient(TOTAL_MONTHS, TARGET_OUTCOMES)}")
print(f"Exact Probability of Target Scenario: {exact_prob:.4f} ({exact_prob * 100:.2f}%)")
print("-" * 38 + "\n")


# --- 2. MONTE CARLO SIMULATION (Risk Desk Validation) ---
# Quants use simulations to test more complex features (like barriers or early calls)

def run_monte_carlo(simulations=100_000):
    regimes = ['Bull', 'Sideways', 'Bear']
    successful_sims = 0

    for _ in range(simulations):
        # Simulate a 6-month path based on our probabilities
        path = random.choices(regimes, weights=PROBABILITIES, k=TOTAL_MONTHS)

        # Count what actually happened in this simulated path
        bull_count = path.count('Bull')
        sideways_count = path.count('Sideways')
        bear_count = path.count('Bear')

        # Check if it matches our structured note's exact condition
        if [bull_count, sideways_count, bear_count] == TARGET_OUTCOMES:
            successful_sims += 1

    return successful_sims / simulations

# Run simulation with 500,000 paths
sim_paths = 500_000
sim_prob = run_monte_carlo(sim_paths)

print("--- MONTE CARLO RISK SIMULATION ---")
print(f"Simulated Paths: {sim_paths:,}")
print(f"Simulated Probability: {sim_prob:.4f} ({sim_prob * 100:.2f}%)")
print(f"Pricing Discrepancy: {abs(exact_prob - sim_prob):.6f}")

--- EXPLICIT MATHEMATICAL PRICING ---
Multinomial Coefficient (Unique Paths): 60
Exact Probability of Target Scenario: 0.0810 (8.10%)
--------------------------------------

--- MONTE CARLO RISK SIMULATION ---
Simulated Paths: 500,000
Simulated Probability: 0.0813 (8.13%)
Pricing Discrepancy: 0.000274
